# Discrete Vapor Cavity Model (DVCM) Showcase

This notebook demonstrates the physical and numerical behavior of the **Discrete Vapor Cavity Model (DVCM)** in RTHYM-MOC. It compares DVCM against the **Legacy Clamp** model using a severe valve-closure transient that triggers water-column separation and subsequent cavity collapse.

> **DVCM regression (JSON traces):** [`dvcm_canonical_verification.ipynb`](dvcm_canonical_verification.ipynb) — the tightest source of truth (`tests/dvcm_*_reference.json`, same as pytest). **Physics formulas:** [`dvcm_physical_verification.ipynb`](dvcm_physical_verification.ipynb). This showcase focuses on **Legacy vs DVCM** at a valve (pedagogy; can be slow on Binder).

## Why use DVCM?
- **Legacy Clamp**: Clamps the hydraulic grade line (HGL) at the vapor pressure floor. It prevents non-physical negative pressures but does *not* track the physical size of the vapor pocket, and it completely misses the severe **secondary water hammer** overpressures that occur when the column separation collapses.
- **DVCM**: Solves the physical regime-switching equations for column separation. It tracks the growth and collapse of the vapor pocket volume and resolves the high-intensity secondary pressure spike generated when the separated liquid columns collide.

## 1. Setup and Dependencies

Import the required libraries. RTHYM-MOC is a high-performance C++ solver exposed to Python via pybind11.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rthym_moc as m

# Scenario constants (used for analytical reference lines in later plots)
P_VAPOR_PSI = -14.0
INITIAL_FLOW_GPM = 1500.0
PIPE_DIAMETER_IN = 12.0
VALVE_ELEVATION_FT = 0.0
VALVE_INITIAL_HEAD_FT = 148.0
P1_LENGTH_FT = 1000.0
P2_LENGTH_FT = 40.0
DESIGN_WAVE_SPEED_FT_S = 4000.0
VALVE_CLOSURE_TIME_S = 0.10


def pipe_area_ft2(diameter_in: float) -> float:
    d_ft = diameter_in / 12.0
    return float(np.pi * (d_ft / 2.0) ** 2)


def velocity_from_flow_ft_s(flow_gpm: float, diameter_in: float) -> float:
    return flow_gpm * m.GPM_TO_CFS / pipe_area_ft2(diameter_in)


def adjusted_wave_speed_ft_s(length_ft: float, dt_s: float, design_a: float = DESIGN_WAVE_SPEED_FT_S) -> float:
    n_reaches = max(int(round(length_ft / (design_a * dt_s))), 1)
    return length_ft / (n_reaches * dt_s)


def vapor_head_ft(elevation_ft: float, p_vapor_psi: float) -> float:
    return elevation_ft + p_vapor_psi * m.PSI_TO_FT


def joukowsky_head_rise_ft(velocity_ft_s: float, wave_speed_ft_s: float) -> float:
    """Single-step Joukowsky estimate: ΔH = a·V/g (see Appendix B.7)."""
    return wave_speed_ft_s * velocity_ft_s / m.G_FT_S2


def expected_collapse_head_rise_ft(
    cavity_volume_ft3: float,
    *,
    wave_speed_ft_s: float,
    area_ft2: float,
    dt_s: float,
) -> float:
    """Discrete cavity-collapse estimate (docs/dvcm_timestep_guidance.md)."""
    return wave_speed_ft_s * cavity_volume_ft3 / (m.G_FT_S2 * area_ft2 * dt_s)


print(f"rthym_moc version: {getattr(m, '__version__', 'unknown')}")
print("Analytical helpers loaded (Joukowsky, vapor floor, DVCM collapse estimate).")

## 2. Define the Pipeline Geometry and Simulation Scenarios

We set up a simple pipeline consisting of:
- An upstream reservoir `R1` (constant HGL = 150 ft)
- A long main pipe `P1` (length = 1000 ft, diameter = 12", wave speed = 4000 ft/s)
- A transient valve `V1`
- A short discharge pipe `P2` (length = 40 ft, diameter = 12", wave speed = 4000 ft/s)
- A downstream reservoir `R2` (constant HGL = 140.0 ft)

The initial flow through the system is **1500 GPM** (initial velocity $V_0 \approx 4.25$ ft/s). At $t = 0.09$ seconds, the valve begins a rapid linear closure, slamming completely shut by $t = 0.10$ seconds.

In [ ]:
def build_network():
    """Configure the showcase pipeline and rapid valve-closure schedule."""
    solver = m.MOCSolver()

    r1 = m.NodeInput()
    r1.id = "R1"
    r1.type = "PressureBoundary"
    r1.elevation = 0.0
    r1.head = 150.0

    v1 = m.NodeInput()
    v1.id = "V1"
    v1.type = "Valve"
    v1.elevation = VALVE_ELEVATION_FT
    v1.diameter = PIPE_DIAMETER_IN
    v1.current_setting = 100.0
    v1.head = VALVE_INITIAL_HEAD_FT

    r2 = m.NodeInput()
    r2.id = "R2"
    r2.type = "PressureBoundary"
    r2.elevation = 0.0
    r2.head = 140.0

    solver.add_node(r1)
    solver.add_node(v1)
    solver.add_node(r2)

    p1 = m.PipeInput()
    p1.id = "P1"
    p1.from_node = "R1"
    p1.to_node = "V1"
    p1.length = P1_LENGTH_FT
    p1.diameter = PIPE_DIAMETER_IN
    p1.roughness = 130.0
    p1.flow_gpm = INITIAL_FLOW_GPM

    p2 = m.PipeInput()
    p2.id = "P2"
    p2.from_node = "V1"
    p2.to_node = "R2"
    p2.length = P2_LENGTH_FT
    p2.diameter = PIPE_DIAMETER_IN
    p2.roughness = 130.0
    p2.flow_gpm = INITIAL_FLOW_GPM

    solver.add_pipe(p1)
    solver.add_pipe(p2)

    solver.set_valve_schedule(
        "V1",
        [
            (0.0, 100.0),
            (0.09, 100.0),
            (VALVE_CLOSURE_TIME_S, 0.0),
            (10.0, 0.0),
        ],
    )

    return solver


# Build once here so this cell produces visible confirmation output.
_showcase_solver = build_network()
_v0_fps = velocity_from_flow_ft_s(INITIAL_FLOW_GPM, PIPE_DIAMETER_IN)
_a_p1 = adjusted_wave_speed_ft_s(P1_LENGTH_FT, dt_s=0.01)
_dh_joukowsky = joukowsky_head_rise_ft(_v0_fps, _a_p1)
_h_vapor = vapor_head_ft(VALVE_ELEVATION_FT, P_VAPOR_PSI)

print("Network build: OK")
print("  Topology : R1 ── P1 (1000 ft) ── V1 ── P2 (40 ft) ── R2")
print(f"  Flow     : {INITIAL_FLOW_GPM:.0f} GPM  →  V0 ≈ {_v0_fps:.2f} ft/s at the valve")
print(f"  Valve    : closes {VALVE_CLOSURE_TIME_S:.2f} s  (schedule loaded on V1)")
print(f"  Reference: Joukowsky ΔH ≈ {_dh_joukowsky:.1f} ft  →  H ≈ {VALVE_INITIAL_HEAD_FT + _dh_joukowsky:.1f} ft (single-step, no reflections)")
print(f"  Reference: vapor-pressure HGL floor ≈ {_h_vapor:.1f} ft")
print(f"  MOCSolver ready ({type(_showcase_solver).__name__})")

## 3. Run Simulation 1: Legacy Clamp Mode

In this run, we use the default `LegacyClamp` cavitation model. The legacy clamp is highly stable under coarse timesteps, so we run it with a standard timestep $dt = 0.01$ seconds.

In [ ]:
solver_legacy = build_network()
res_legacy = solver_legacy.run(
    total_time=10.0,
    dt=0.01,
    p_vapor_psi=-14.0,  # ~full vacuum
    cavitation_model=m.CavitationModel.LegacyClamp
)

time_legacy = np.array(res_legacy["time"])
head_legacy = np.array(res_legacy["node_head"]["V1"])
cav_legacy = np.array(res_legacy["node_cavitation"]["V1"])

print("--- Legacy Clamp Results ---")
print(f"Total Steps Simulated: {len(time_legacy)}")
print(f"Maximum HGL Head at Valve: {np.max(head_legacy):.2f} ft")
print(f"Minimum HGL Head at Valve: {np.min(head_legacy):.2f} ft")
print(f"Cavitating Steps: {np.sum(cav_legacy)}")

## 4. Run Simulation 2: DVCM Mode

Now we opt-in to the `DVCM` model.

> **Important:** Because DVCM tracks physical cavity volumes that can change rapidly during collapse, we must reduce the timestep to prevent numerical volume integration overshoot. We use **$dt = 0.0001$ seconds** to ensure numerical stability.

In [ ]:
solver_dvcm = build_network()
res_dvcm = solver_dvcm.run(
    total_time=10.0,
    dt=0.0001,
    p_vapor_psi=-14.0,  # ~full vacuum
    cavitation_model=m.CavitationModel.DVCM
)

time_dvcm = np.array(res_dvcm["time"])
head_dvcm = np.array(res_dvcm["node_head"]["V1"])
cav_active_dvcm = np.array(res_dvcm["node_cavity_active"]["V1"])
cav_volume_dvcm = np.array(res_dvcm["node_cavity_volume"]["V1"])
collapse_counts_dvcm = np.array(res_dvcm["node_cavity_collapse_count"]["V1"])

print("--- DVCM Results ---")
print(f"Total Steps Simulated: {len(time_dvcm)}")
print(f"Maximum HGL Head at Valve: {np.max(head_dvcm):.2f} ft")
print(f"Minimum HGL Head at Valve: {np.min(head_dvcm):.2f} ft")
print(f"Active Cavitation Steps: {np.sum(cav_active_dvcm)}")
print(f"Max Cavity Volume: {np.max(cav_volume_dvcm):.4f} ft³")
print(f"Total Cavity Collapses: {collapse_counts_dvcm[-1]}")

## 5. Visual Comparisons Against References

The plots overlay three sources of truth on the simulated valve head:

1. **Joukowsky (analytical)** — single-step surge $\Delta H = a V_0 / g$ from Appendix B.7 (does not include pipe reflections).
2. **Vapor-pressure floor (analytical)** — HGL clamp corresponding to $P_{\text{vap}} = -14$ psi gauge at the valve elevation.
3. **DVCM collapse estimate (analytical)** — discrete water-column collision $\Delta H \approx a' V / (g A \Delta t)$ at the primary cavity-collapse event (see `docs/dvcm_timestep_guidance.md`).

Legacy Clamp and DVCM traces are the numerical models under comparison.

In [ ]:
area_ft2 = pipe_area_ft2(PIPE_DIAMETER_IN)
h_vapor = vapor_head_ft(VALVE_ELEVATION_FT, P_VAPOR_PSI)
h_joukowsky_peak = VALVE_INITIAL_HEAD_FT + joukowsky_head_rise_ft(_v0_fps, _a_p1)

# Primary DVCM collapse (first event with reasonable V_before)
collapse_flag = np.asarray(res_dvcm["node_cavity_collapse_flag"]["V1"], dtype=int)
volume_dvcm = np.asarray(cav_volume_dvcm, dtype=float)
cap_ft3 = 0.5 * area_ft2 * P1_LENGTH_FT + 0.5 * area_ft2 * P2_LENGTH_FT
collapse_step = None
for step in np.flatnonzero(collapse_flag):
    v_before = float(volume_dvcm[step - 1])
    if 0.0 < v_before <= 0.25 * cap_ft3:
        collapse_step = int(step)
        break

dt_dvcm = 0.0001
a_p2 = adjusted_wave_speed_ft_s(P2_LENGTH_FT, dt_dvcm)
if collapse_step is not None:
    v_before = float(volume_dvcm[collapse_step - 1])
    dh_collapse_expected = expected_collapse_head_rise_ft(
        v_before, wave_speed_ft_s=a_p2, area_ft2=area_ft2, dt_s=dt_dvcm
    )
    t_collapse = float(time_dvcm[collapse_step])
    h_collapse_expected = h_vapor + dh_collapse_expected
    win = slice(collapse_step, min(collapse_step + 30, len(head_dvcm)))
    dh_collapse_observed = float(head_dvcm[win].max()) - h_vapor
else:
    dh_collapse_expected = np.nan
    h_collapse_expected = np.nan
    dh_collapse_observed = np.nan
    t_collapse = np.nan

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=False)

for ax, tmax, title_suffix in (
    (axes[0], 1.0, "early transient (0–1 s)"),
    (axes[1], 10.0, "full window (0–10 s)"),
):
    ax.plot(time_legacy, head_legacy, label="Legacy Clamp", color="gray", alpha=0.75, linestyle="--")
    ax.plot(time_dvcm, head_dvcm, label="DVCM", color="crimson", linewidth=1.2)
    ax.axhline(h_vapor, color="royalblue", linestyle=":", linewidth=1.5, label=f"Vapor floor ({h_vapor:.1f} ft)")
    ax.axhline(
        h_joukowsky_peak,
        color="seagreen",
        linestyle="-.",
        linewidth=1.5,
        label=f"Joukowsky single-step ({h_joukowsky_peak:.0f} ft)",
    )
    if collapse_step is not None:
        ax.axhline(
            h_collapse_expected,
            color="darkorange",
            linestyle=":",
            linewidth=1.2,
            label=f"DVCM collapse est. ({h_collapse_expected:.0f} ft)",
        )
        ax.axvline(t_collapse, color="darkorange", alpha=0.35, linewidth=1.0)
    ax.axvline(VALVE_CLOSURE_TIME_S, color="navy", linestyle=":", alpha=0.4)
    ax.set_xlim(0.0, tmax)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("HGL at valve (ft)")
    ax.set_title(f"Valve head vs analytical references — {title_suffix}")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)

fig.text(0.01, 0.01, "Vertical line: valve fully closed", fontsize=9, color="navy")
fig.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison table (simulation vs analytical references)
idx_legacy_peak = int(np.argmax(head_legacy[time_legacy <= 1.0]))
idx_dvcm_peak_early = int(np.argmax(head_dvcm[time_dvcm <= 1.0]))

rows = [
    ("Vapor HGL floor", h_vapor, np.min(head_legacy), np.min(head_dvcm)),
    (
        "Early peak head (t ≤ 1 s)",
        h_joukowsky_peak,
        head_legacy[idx_legacy_peak],
        head_dvcm[idx_dvcm_peak_early],
    ),
    (
        "Global max head",
        np.nan,
        np.max(head_legacy),
        np.max(head_dvcm),
    ),
]

if collapse_step is not None:
    rows.append(
        (
            f"Post-collapse ΔH @ t={t_collapse:.3f}s",
            dh_collapse_expected,
            np.nan,
            dh_collapse_observed,
        )
    )

print(f"{'Metric':<32} {'Reference':>12} {'Legacy':>12} {'DVCM':>12}")
print("-" * 72)
for name, ref, leg, dv in rows:
    ref_s = f"{ref:12.2f}" if np.isfinite(ref) else f"{'—':>12}"
    leg_s = f"{leg:12.2f}" if np.isfinite(leg) else f"{'—':>12}"
    dv_s = f"{dv:12.2f}" if np.isfinite(dv) else f"{'—':>12}"
    print(f"{name:<32} {ref_s} {leg_s} {dv_s}")

if collapse_step is not None and np.isfinite(dh_collapse_expected):
    rel_err = abs(dh_collapse_observed - dh_collapse_expected) / dh_collapse_expected
    print(f"\nDVCM collapse ΔH vs discrete estimate: relative error = {rel_err:.1%}")

### Analysis of the Head Comparison

1. **Joukowsky reference (green dash-dot)** — The single-step estimate uses pre-closure velocity and the P1 Courant-adjusted wave speed. Simulated early peaks can exceed this level because reflected waves from the 1000 ft main and 40 ft stub superpose at the valve (see Appendix B.7 for a cross-engine study of this closure geometry class).

2. **Vapor floor (blue dotted)** — Both models clamp at the analytical floor ($P_{\text{vap}} = -14$ psi gauge). Legacy Clamp cannot produce the secondary spike when the cavity collapses.

3. **DVCM collapse reference (orange dotted)** — When the tracked cavity volume returns to zero, the discrete collision formula predicts an additional head rise above the vapor floor. Only the DVCM trace reaches (and often exceeds) this secondary reference; Legacy Clamp stays at the floor through separation.

4. **Separation duration** — DVCM sustains the low-pressure, finite-volume state longer than Legacy Clamp because it integrates cavity growth while the columns separate, then releases a collapse spike when they reunite.

## 6. Cavity Volume Growth and Collapse

Let's inspect the growth and collapse of the physical vapor cavity volume at the valve as calculated by DVCM.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Panel 1: cavity volume with capacity cap (reference bound)
axes[0].plot(time_dvcm, cav_volume_dvcm, color="darkcyan", linewidth=2, label="DVCM cavity volume")
axes[0].axhline(cap_ft3, color="black", linestyle=":", label=f"Capacity cap ({cap_ft3:.2f} ft³)")
if collapse_step is not None:
    axes[0].axvline(t_collapse, color="darkorange", linestyle="--", alpha=0.7, label="Primary collapse")
axes[0].set_ylabel("Cavity volume (ft³)")
axes[0].set_title("DVCM cavity volume vs physical capacity")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right")

# Panel 2: valve head with vapor floor for context
axes[1].plot(time_dvcm, head_dvcm, color="crimson", linewidth=1.2, label="DVCM valve head")
axes[1].axhline(h_vapor, color="royalblue", linestyle=":", label=f"Vapor floor ({h_vapor:.1f} ft)")
if collapse_step is not None:
    axes[1].axhline(h_collapse_expected, color="darkorange", linestyle=":", label="Collapse head estimate")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("HGL at valve (ft)")
axes[1].set_xlim(0, 6.0)
axes[1].set_title("Head response aligned with cavity collapse")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

fig.tight_layout()
plt.show()

print(f"Max cavity volume: {np.max(cav_volume_dvcm):.4f} ft³  ({100*np.max(cav_volume_dvcm)/cap_ft3:.1f}% of cap)")
print(f"Total collapse events: {int(collapse_counts_dvcm[-1])}")

### Interpretation of Cavity Dynamics

- **Capacity bound (dotted black)** — DVCM limits cavity volume to the adjacent pipe storage fraction; growth should remain below this curve.
- **Initiation** — Volume becomes non-zero once the valve head reaches the vapor floor and the regime switches to active cavity tracking.
- **Collapse marker** — The orange vertical line is the first primary collapse used for the analytical $\Delta H$ comparison in §5.
- **Head panel** — The secondary surge above the vapor floor should track the orange collapse estimate more closely than Legacy Clamp (which omits cavity-volume physics entirely).

## 7. Numerical Stability and Timestep Guidance

The DVCM model is physically realistic but is highly sensitive to the timestep size $dt$.
- **Legacy Clamp** has no state integration, making it stable under coarse grids (e.g. $dt = 0.01$ s).
- **DVCM** requires tracking a small volume over time:
  $$V_c(t) = \int (Q_{\text{out}} - Q_{\text{in}}) \, dt$$
  If $dt$ is too large, the integrated volume can overshoot or chatter, resulting in numerical instabilities (causing `NaN`/`Inf` propagation or negative volumes).
- **Recommendation**: Always use **$dt \le 0.001$ s** when `DVCM` is active (recommend **$0.0001$ s** or smaller for severe column separation events).

---

## 8. Launching in Binder

This repository is configured for Binder, which makes it easy to run and explore this notebook in your web browser without installing anything locally.

To launch this notebook in Binder:

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=examples%2Fdvcm_showcase.ipynb)

Or manually: open [mybinder.org](https://mybinder.org), paste `https://github.com/jlillywh/RTHYM-MOC`, select branch `main`, and open `examples/dvcm_showcase.ipynb`.